# Predicting Credit Card Fraud with TabPFN-3.5

*Playground Series — Season 6, Episode 11*

This notebook tackles this competition using **[TabPFN](https://github.com/PriorLabs/TabPFN)**, a foundation model for tabular data.

There is **no feature engineering** and practically **no hyperparameter tuning** below. We read the CSV, hand the raw frame to TabPFN, and predict. Categorical columns, missing values, and scaling are all handled inside the model, and predictions come out of a single forward pass: no gradient boosting to tune, no hyperparameter search.

### What is TabPFN?

TabPFN (Tabular Prior-data Fitted Network) is a transformer pre-trained on millions of synthetic tabular datasets. Rather than training from scratch on your data, it performs *in-context learning*: your training rows go in as context and predictions come straight out. TabPFN-3 scales to one million rows.

| | |
|---|---|
| **Docs** | https://docs.priorlabs.ai |
| **GitHub** | https://github.com/PriorLabs/TabPFN |
| **Hugging Face** | https://huggingface.co/Prior-Labs/tabpfn_3 |
| **Prior Labs** | https://priorlabs.ai |
| **Install** | `pip install tabpfn` |

## Setup

Install TabPFN. The checkpoint (TabPFN-3.5) is mounted in this notebook's inputs. Accept the license on the model page: https://www.kaggle.com/models/prior-labsai/tabpfn-3-5/

In [1]:
!uv pip install tabpfn --system -q

In [2]:
import os
os.environ["TABPFN_MODEL_CACHE_DIR"] = "/kaggle/input/models/prior-labsai/tabpfn-3-5/pytorch/default/1"

import numpy as np
import pandas as pd

DATA_DIR = "/kaggle/input/competitions/playground-series-s6e11"
TARGET = "is_fraud"

## Load the data

Load the training and test sets and drop the `id` column, which carries no signal.

In [3]:
df = pd.read_csv(f"{DATA_DIR}/train.csv", index_col="id")
test = pd.read_csv(f"{DATA_DIR}/test.csv", index_col="id")

print(f"Training set: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Test set: {test.shape[0]:,} rows")
print(f"Fraud rate: {df[TARGET].mean():.4f}")
df.head()

Training set: 699,960 rows x 25 columns
Test set: 299,984 rows
Fraud rate: 0.0167


,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,time_of_day_hour,...,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,is_fraud
id,,,,,,,,,,,,,,,,,,,,,
0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,17,...,OTP,POS,iPhone,False,False,False,False,False,False,0.0
1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,23,...,PIN,In-App,Android Phone,False,False,False,False,False,False,0.0
2,6.36,1.20,4,8.17,50,55,1046.88,0,35.8,15,...,OTP,Online,Tablet,False,False,True,False,False,False,0.0
3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,7,...,3D Secure,POS,Android Phone,False,False,False,False,False,False,0.0
4,653.85,0.80,4,11.69,37,52,3291.06,0,34.8,4,...,3D Secure,Online,Windows PC,False,False,False,False,False,False,0.0


## Why TabPFN here?

A few properties make TabPFN a strong fit for a task like this:

- **Minimal preprocessing.** The categorical columns and any missing values are handled internally; we pass the raw frame as-is.
- **No training loop.** "Fitting" is essentially loading your data as context. There is no epoch tuning, early stopping, or learning-rate schedule.
- **Strong out-of-the-box accuracy.** TabPFN is designed to be competitive with heavily tuned gradient-boosted trees with little or no tuning.
- **Calibrated probabilities.** The competition is scored by ROC AUC, so we submit the predicted fraud probability directly.

Separate the target from the features:

In [4]:
y = df.pop(TARGET)

## Fit TabPFN

All 700k training rows as one context do not fit into the 16 GB of a T4, so we let TabPFN subsample the context per estimator.

Notes on the configuration:

- **`device=["cuda:0", "cuda:1"]`** runs across both GPUs of Kaggle's T4 ×2 accelerator, one estimator per GPU.
- **`n_estimators=2`** keeps inference fast on this large test set; raising it trades runtime for a small potential accuracy bump.
- **`SUBSAMPLE_SAMPLES=350_000`** gives each estimator its own 350k-row context.
- **`SAMPLE_SUBSAMPLING_METHOD="majority_downsample"`** keeps every fraud row in each context and fills the rest with non-fraud rows. TabPFN corrects the predicted probabilities for the shifted class prior; for binary targets the correction keeps the ranking, so ROC AUC is unaffected.
- **`fit_mode="fit_with_cache"`** computes the training context once and reuses it for every test chunk.

In [5]:
%%time

from tabpfn import TabPFNClassifier

clf = TabPFNClassifier(
    device=["cuda:0", "cuda:1"],
    n_estimators=5,
    fit_mode="fit_with_cache",
    inference_config={
        "SUBSAMPLE_SAMPLES": 400_000,
        "SAMPLE_SUBSAMPLING_METHOD": "majority_downsample",
    },
)
clf.fit(df, y)

CPU times: user 1h 28min 30s, sys: 49min 23s, total: 2h 17min 54s
Wall time: 1h 25min 51s


TabPFNClassifier(device=['cuda:0', 'cuda:1'], fit_mode='fit_with_cache',
                 inference_config={'SAMPLE_SUBSAMPLING_METHOD': 'majority_downsample',
                                   'SUBSAMPLE_SAMPLES': 400000},
                 n_estimators=5)

## Predict on the test set

Predict the fraud probability in chunks of 100k test rows to bound GPU memory.

In [6]:
CHUNK_SIZE = 100_000
fraud_col = list(clf.classes_).index(1)
predictions = np.concatenate([
    clf.predict_proba(test.iloc[i : i + CHUNK_SIZE])[:, fraud_col]
    for i in range(0, len(test), CHUNK_SIZE)
])

## Create the submission

In [7]:
sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
assert (sub["id"].to_numpy() == test.index.to_numpy()).all()
sub[TARGET] = predictions
sub.to_csv("submission.csv", index=False)
sub.head()

,id,is_fraud
0,699960,0.000152
1,699961,0.000034
2,699962,0.074097
3,699963,0.001495
4,699964,0.000923
